# BML vs NaSch: Side-by-Side Comparison

Both models are cellular automaton traffic models, yet they produce qualitatively
different behaviour.  This notebook places them side by side on three questions:

| Section | Question |
|---|---|
| 1 | Fundamental diagram: smooth hump vs sharp drop? |
| 2 | Congestion type: stop-and-go waves vs total gridlock? |
| 3 | Role of randomness: is stochasticity necessary for complex behaviour? |
| 4 | Summary table |

Both models are re-implemented here so the notebook is self-contained.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

# ── BML model ─────────────────────────────────────────────────────────────
class BML:
    def __init__(self, L, density, r_horiz=0.5, seed=None):
        self.L = L
        self.t = 0
        rng    = np.random.default_rng(seed)
        n_total = int(round(density * L * L))
        n_horiz = int(round(r_horiz * n_total))
        n_vert  = n_total - n_horiz
        flat      = rng.permutation(L * L)
        self.grid = np.zeros(L * L, dtype=np.int8)
        self.grid[flat[:n_horiz]]                 = 1
        self.grid[flat[n_horiz:n_horiz + n_vert]] = 2
        self.grid = self.grid.reshape(L, L)

    def step(self):
        n_total = int((self.grid > 0).sum())
        for axis, car_type, roll_check, roll_dest, dest_val in [
            (1, 1, -1, 1, 1),   # horizontal: check right, move right
            (0, 2,  1, -1, 2),  # vertical:   check up,    move up
        ]:
            can_move = (self.grid == car_type) & (np.roll(self.grid, roll_check, axis=axis) == 0)
            if can_move.any():
                g = self.grid.copy()
                g[can_move] = 0
                g[np.roll(can_move, roll_dest, axis=axis)] = dest_val
                self.grid = g
        self.t += 1

    def flow(self, T=200):
        flows = []
        for _ in range(T):
            n_tot = int((self.grid > 0).sum())
            mh = int(((self.grid == 1) & (np.roll(self.grid, -1, axis=1) == 0)).sum())
            mv = int(((self.grid == 2) & (np.roll(self.grid,  1, axis=0) == 0)).sum())
            self.step()
            flows.append((mh + mv) / max(n_tot, 1))
        return float(np.mean(flows))

    def row_occupancy(self, row, T=200):
        frames = np.zeros((T, self.L), dtype=np.int8)
        for t in range(T):
            self.step()
            frames[t] = (self.grid[row] > 0).astype(np.int8)
        return frames


# ── NaSch periodic model ───────────────────────────────────────────────────
class NaSch_Periodic:
    def __init__(self, L=200, n_cars=50, v_max=5, p_rand=0.3, seed=None):
        self.L      = L
        self.v_max  = v_max
        self.p_rand = p_rand
        rng = np.random.default_rng(seed)
        self.road = np.full(L, -1, dtype=int)
        positions = rng.choice(L, n_cars, replace=False)
        self.road[positions] = rng.integers(0, v_max + 1, n_cars)

    def step(self):
        road   = self.road
        L      = self.L
        v_max  = self.v_max
        p_rand = self.p_rand
        car_pos = np.where(road >= 0)[0]
        if len(car_pos) == 0:
            return
        velocities = road[car_pos].copy()
        velocities = np.minimum(velocities + 1, v_max)
        # gap to next car (periodic)
        gaps = np.empty(len(car_pos), dtype=int)
        for i, p in enumerate(car_pos):
            g = 0
            for j in range(1, L):
                if road[(p + j) % L] >= 0:
                    break
                g += 1
            gaps[i] = g
        velocities = np.minimum(velocities, gaps)
        velocities = np.maximum(velocities, 0)
        rnd = np.random.default_rng()
        rand_mask = rnd.random(len(car_pos)) < p_rand
        velocities[rand_mask] = np.maximum(velocities[rand_mask] - 1, 0)
        new_road = np.full(L, -1, dtype=int)
        new_pos  = (car_pos + velocities) % L
        new_road[new_pos] = velocities
        self.road = new_road

    def mean_flow(self):
        car_pos = np.where(self.road >= 0)[0]
        if len(car_pos) == 0:
            return 0.0
        return float(np.mean(self.road[car_pos])) * len(car_pos) / self.L

    def occupancy(self, T=200):
        frames = np.zeros((T, self.L), dtype=int)
        for t in range(T):
            frames[t] = (self.road >= 0).astype(int)
            self.step()
        return frames


print('Models defined.')

## Section 1: Fundamental Diagram

The fundamental diagram (flow vs density) is the canonical summary of a traffic model.

- **NaSch** produces a smooth hump: flow rises, peaks, then falls.  
  The peak and shape are controlled by $p_{\text{rand}}$ and $v_{\text{max}}$.
- **BML** produces an abrupt drop from high flow to zero at $\rho_c$.  
  There is no intermediate congested phase — only free flow or gridlock.

Note: BML flow is measured as the fraction of cars that move per step,
which is comparable to the NaSch normalised flow $J / J_{\text{max}}$.

In [ ]:
# ── Section 1: Fundamental diagrams ───────────────────────────────────────
T_WARM   = 500
T_MEAS   = 300
L_BML    = 100
L_NASCH  = 200

# BML sweep
rhos_bml  = np.linspace(0.02, 0.70, 35)
flows_bml = []
for i, rho in enumerate(rhos_bml):
    sim = BML(L=L_BML, density=rho, seed=i)
    for _ in range(T_WARM):
        sim.step()
    flows_bml.append(sim.flow(T_MEAS))

# NaSch sweep (p_rand = 0.3, v_max = 5)
rhos_nasch  = np.linspace(0.01, 0.99, 35)
flows_nasch = []
for rho in rhos_nasch:
    n_cars = max(1, int(rho * L_NASCH))
    sim    = NaSch_Periodic(L=L_NASCH, n_cars=n_cars, v_max=5, p_rand=0.3, seed=0)
    for _ in range(300):
        sim.step()
    J = sum(sim.mean_flow() for _ in [sim.step() or 1 for _ in range(T_MEAS)]) / T_MEAS
    # redo properly
    sim = NaSch_Periodic(L=L_NASCH, n_cars=n_cars, v_max=5, p_rand=0.3, seed=0)
    for _ in range(300):
        sim.step()
    J_vals = []
    for _ in range(T_MEAS):
        J_vals.append(sim.mean_flow())
        sim.step()
    flows_nasch.append(float(np.mean(J_vals)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rhos_nasch, flows_nasch, 'o-', color='steelblue', lw=2, ms=4)
axes[0].set_xlabel('Density \u03c1')
axes[0].set_ylabel('Mean flow  J')
axes[0].set_title(f'NaSch  ($v_{{max}}=5$, $p=0.3$, $L={L_NASCH}$)')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 1)

axes[1].plot(rhos_bml, flows_bml, 'o-', color='tomato', lw=2, ms=4)
axes[1].set_xlabel('Density \u03c1')
axes[1].set_ylabel('Mean flow  (fraction of cars moving per step)')
axes[1].set_title(f'BML  ($L={L_BML}$, $r_{{horiz}}=0.5$)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 1)

fig.suptitle('Fundamental Diagram Comparison', fontsize=13)
plt.tight_layout()
plt.show()
print('NaSch: smooth hump.  BML: sharp step (discontinuous phase transition).')

## Section 2: Nature of Congestion

Space-time diagrams reveal the *structure* of congestion, not just its existence.

- **NaSch congested**: backward-propagating stop-and-go waves — diagonal bands
  tilting upper-left as jams travel upstream while cars travel downstream.
- **BML gridlock**: a single row of the 2D grid simply **freezes**.  No wave
  structure — just a static, permanently blocked configuration.

This is a qualitative difference: NaSch congestion is dynamic; BML gridlock is static.

In [ ]:
# ── Section 2: Space-time diagrams ────────────────────────────────────────
T_REC = 200

# NaSch free flow
ns_ff = NaSch_Periodic(L=200, n_cars=20, v_max=5, p_rand=0.3, seed=1)
for _ in range(300): ns_ff.step()
st_ns_ff = ns_ff.occupancy(T_REC)

# NaSch congested
ns_cg = NaSch_Periodic(L=200, n_cars=140, v_max=5, p_rand=0.3, seed=2)
for _ in range(300): ns_cg.step()
st_ns_cg = ns_cg.occupancy(T_REC)

# BML free flow
bml_ff = BML(L=100, density=0.10, seed=3)
for _ in range(500): bml_ff.step()
st_bml_ff = bml_ff.row_occupancy(row=50, T=T_REC)

# BML gridlock
bml_gl = BML(L=100, density=0.50, seed=4)
for _ in range(500): bml_gl.step()
st_bml_gl = bml_gl.row_occupancy(row=50, T=T_REC)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
titles = [
    'NaSch: free flow\n($\\rho\\approx0.10$)',
    'NaSch: congested\n($\\rho\\approx0.70$)',
    'BML: free flow\n($\\rho=0.10$, row 50)',
    'BML: gridlock\n($\\rho=0.50$, row 50)',
]
for ax, data, title in zip(axes, [st_ns_ff, st_ns_cg, st_bml_ff, st_bml_gl], titles):
    ax.imshow(data, cmap='binary', aspect='auto', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Position')
    ax.set_ylabel('Time (steps)')

fig.suptitle('Space\u2013Time Diagrams: Congestion Structure (black = occupied)', fontsize=13)
plt.tight_layout()
plt.show()
print('NaSch congested: note backward-tilted bands = stop-and-go waves.')
print('BML gridlock: row freezes completely — no wave structure.')

## Section 3: Role of Randomness

NaSch attributes its complex behaviour to the stochastic dawdling step ($p_{\text{rand}}$).
Setting $p_{\text{rand}} = 0$ makes NaSch fully deterministic — and the model then
produces only free flow at low density and perfect congestion at high density,
with no stop-and-go waves.

BML is already deterministic ($p_{\text{rand}} = 0$ by construction), yet still
produces a sharp phase transition and self-organised stripe patterns.

This section compares fundamental diagrams for:
- NaSch with $p_{\text{rand}} = 0$ (deterministic NaSch)
- NaSch with $p_{\text{rand}} = 0.3$ (standard)
- BML (always deterministic)

In [ ]:
# ── Section 3: Role of randomness ─────────────────────────────────────────
rhos_r = np.linspace(0.01, 0.99, 30)

def nasch_fd(p_rand, L=200, rhos=None, T_warm=300, T_meas=200):
    if rhos is None:
        rhos = np.linspace(0.01, 0.99, 30)
    flows = []
    for rho in rhos:
        n_cars = max(1, int(rho * L))
        sim = NaSch_Periodic(L=L, n_cars=n_cars, v_max=5, p_rand=p_rand, seed=0)
        for _ in range(T_warm): sim.step()
        J_vals = []
        for _ in range(T_meas):
            J_vals.append(sim.mean_flow())
            sim.step()
        flows.append(float(np.mean(J_vals)))
    return np.array(flows)

flows_det  = nasch_fd(p_rand=0.0, rhos=rhos_r)
flows_stoch = nasch_fd(p_rand=0.3, rhos=rhos_r)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rhos_r,    flows_det,   's--', color='seagreen',   lw=2, ms=5, label='NaSch  $p=0$ (deterministic)')
ax.plot(rhos_r,    flows_stoch, 'o-',  color='steelblue',  lw=2, ms=5, label='NaSch  $p=0.3$ (stochastic)')
ax.plot(rhos_bml,  flows_bml,   '^-',  color='tomato',     lw=2, ms=5, label='BML  (deterministic, 2D)')
ax.set_xlabel('Density \u03c1')
ax.set_ylabel('Mean flow')
ax.set_title('Role of Randomness: Deterministic vs Stochastic Models')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()
print('NaSch p=0: triangular diagram (no stop-and-go, no hump below peak).')
print('NaSch p=0.3: smooth hump — randomness lowers capacity but creates waves.')
print('BML: sharp drop — determinism + 2D geometry alone produce a phase transition.')

## Section 4: Summary

| Property | NaSch ($p=0.3$) | NaSch ($p=0$) | BML |
|---|---|---|---|
| Dimension | 1-D | 1-D | 2-D |
| Stochastic? | Yes | No | No |
| Fundamental diagram | Smooth hump | Triangular | Sharp drop |
| Congestion type | Stop-and-go waves | Compact jam | Total gridlock |
| Congestion reversible? | Yes | Yes | **No** (absorbing state) |
| Key mechanism | Dawdling cascade | Speed matching | Geometric frustration |

In [ ]:
# ── Section 4: Overlay summary ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rhos_r,   flows_det,   's--', color='seagreen',  lw=2, ms=5, label='NaSch  $p=0$ (deterministic 1D)')
ax.plot(rhos_r,   flows_stoch, 'o-',  color='steelblue', lw=2, ms=5, label='NaSch  $p=0.3$ (stochastic 1D)')
ax.plot(rhos_bml, flows_bml,   '^-',  color='tomato',    lw=2, ms=5, label='BML (deterministic 2D)')
ax.set_xlabel('Density \u03c1', fontsize=12)
ax.set_ylabel('Mean flow', fontsize=12)
ax.set_title('All Three Models: Fundamental Diagram', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()
print('Key takeaway: geometry (2D) alone can produce a sharper phase transition than stochasticity.')